In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score
from imblearn.over_sampling import RandomOverSampler, ADASYN, SMOTE
import optuna
from joblib import Parallel, delayed
from tqdm import tqdm
import random
import warnings
from optuna.exceptions import TrialPruned

import os

# Limit each parallel process to one thread per library
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['TORCH_NUM_THREADS'] = '1'  # For PyTorch

# Additional control for PyTorch if needed
import torch
torch.set_num_threads(1)

# # For reproducibility
# def set_seed(seed=42):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)

# set_seed()

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Load the dataset
data = pd.read_excel("class123_dataset.xlsx")

# Define feature groups (as provided)
feature_groups = {
    'Genotype': [
        'rs11225395', 'rs1144393', 'rs650108', 'rs591058', 'rs2252070', 'rs4986938', 'rs1800012', 'rs4789932', 'rs9340799', 'rs970547', 
        'rs1800795', 'rs13946', 'rs12722', 'class1_SNP_risk_score', 'rs7528684', 'rs4919510', 'rs1937810', 'rs6481512', 'rs1249269', 
        'rs12574452', 'rs12429486', 'rs4454832', 'rs2761884', 'rs62051384', 'rs4362400', 'rs2586488', 'rs2277698', 'rs1045485', 
        'rs143383', 'rs17576', 'rs2305948', 'rs1011814', 'rs11154027', 'rs2234693', 'rs1643821', 'rs2010963', 'rs10263021', 'rs149047058', 
        'rs420257', 'rs42517', 'rs42522', 'rs42531', 'rs413826', 'rs2104772', 'rs1330363', 'class12_SNP_risk_score', 'rs3753841', 
        'rs57104447', 'rs1887632', 'rs4654760', 'rs1137101', 'rs2306033', 'rs2277268', 'rs4988321', 'rs11232681', 'rs1718119', 'rs3751143', 
        'rs1544410', 'rs2228570', 'rs4328262', 'rs1021188', 'rs74544784', 'rs78391032', 'rs77569527', 'rs117544024', 'rs912336', 
        'rs3218791', 'rs911263', 'rs2525504', 'rs17756404', 'rs4903399', 'rs10132091', 'rs17583842', 'rs1676303', 'rs11629171', 
        'rs2281518', 'rs2285053', 'rs71404070', 'rs710079', 'rs2858056', 'rs820218', 'rs3018362', 'rs1800470', 'rs1800469', 'rs25487', 
        'rs25489', 'rs2289360', 'rs183364169', 'rs11177', 'rs6617', 'rs3219008', 'rs13107325', 'rs60713544', 'rs145648292', 'rs4244032', 
        'rs12656106', 'rs3045', 'rs187483', 'rs4701616', 'rs144414988', 'rs1800629', 'rs10484958', 'rs4730153', 'rs1800797', 'rs1554606', 
        'rs2237352', 'rs4725069', 'rs12154667', 'rs1548456', 'rs3216902', 'rs35360670', 'rs13317', 'rs1800972', 'rs7035322', 'rs7021589', 
        'rs72758637', 'rs10759753', 'rs3789870', 'rs1138545', 'rs3196378', 'rs1134170', 'rs10992075', 'rs1590', 'rs144371252', 
        'rs761804508', 'class123_SNP_risk_score', 'sex'
    ],
    'History': [
        'Age', 'lower_limb_days_total', 'average_run_hours', 'average_interval_training_frequency', 'EDEQ_total', 'tracking_period_injury',
        'past_stress_injury', 'LEAF-Q', 'Athlete_Score', 'average_run_frequency', 'past_month_injury'
    ],
    'Phenotype': [
        'hip_abduction_peak_torque', 'total_ad_ab_ratio', 'knee_extension_peak_torque', 'knee_flexion_peak_torque', 'navicular_drop', 
        'navicular_drop_asymmetry', 'Q_angle', 'Q_angle_asymmetry', 'VALR_12', 'Impact_peak_12', 'Duty_factor_12', 'BMI', 'BMD_spine',
        'hip_abduction_peak_torque_asymmetry', 'hip_adduction_peak_torque', 'hip_adduction_peak_torque_asymmetry', 
        'knee_extension_peak_torque_asymmetry', 'knee_flexion_peak_torque_asymmetry', 'total_fl_ex_ratio', 'leg_lean_mass', 
        'hip_abduction_peak_angle', 'hip_abduction_peak_angle_asymmetry', 'hip_adduction_peak_angle', 'hip_adduction_peak_angle_asymmetry', 
        'ad_ab_ratio_asymmetry', 'knee_extension_peak_angle', 'knee_extension_peak_angle_asymmetry', 'knee_flexion_peak_angle', 
        'knee_flexion_peak_angle_asymmetry', 'fl_ex_ratio_asymmetry', 'VILR_10', 'VALR_10', 'VILR_asymmetry_10', 'VALR_asymmetry_10', 
        'Impact_peak_10', 'Impact_peak_asymmetry_10', 'Flight_time_10', 'Contact_time_10', 'Duty_factor_10', 'Step_frequency_10', 
        'Cadence_asymmetry_10', 'Duty_factor_asymmetry_10', 'VILR_12', 'VILR_asymmetry_12', 'VALR_asymmetry_12', 
        'Impact_peak_asymmetry_12', 'Flight_time_12', 'Contact_time_12', 'Step_frequency_12', 'Cadence_asymmetry_12', 
        'Duty_factor_asymmetry_12', 'Alt_strike', 'height', 'Mass', 'thigh_lean_mass', 'thigh_ffmi', 'lower_leg_lean_mass', 
        'lower_leg_ffmi', 'leg_ffmi', 'total_lean_mass', 'total_ffmi', 'calf_size', 'BMD_hip', 'BMD_body',
    ],
    'Behaviour': [
        'fat_intake_avg', 'past_month_distance', 'past_month_ratio', 'SC_past_season', 'non_running_past_season', 'fat_intake_BW', 
        'fat_percentage_avg', 'average_energy_availability', 'protein_intake_BW', 'omega3_intake_BW', 'vitaminD_intake_BW', 
        'vitaminC_intake_BW', 'vitaminE_intake_BW', 'calcium_intake_BW', 'copper_intake_BW', 'iron_intake_BW', 'glycine_intake_BW', 
        'arginine_intake_BW', 'past_month_min', 'past_week_ratio', 'past_month_volume_low', 'past_week_ratio_low', 'past_month_ratio_low', 
        'past_month_volume_moderate', 'past_week_ratio_moderate', 'past_month_ratio_moderate', 'past_month_volume_high', 
        'past_week_ratio_high', 'past_month_ratio_high', 'past_month_volume_very_high', 'past_week_ratio_very_high', 
        'past_month_ratio_very_high', 'past_month_calculated_volume', 'past_week_ratio_calculated_volume', 
        'past_month_ratio_calculated_volume', 'resistance_training_past_month', 'resistance_training_past_season', 
        'bodyweight_exercises_past_month', 'bodyweight_exercises_past_season', 'core_stability_past_month', 'core_stability_past_season', 
        'balance_training_past_month', 'balance_training_past_season', 'plyometrics_past_month', 'plyometrics_past_season', 
        'drills_past_month', 'drills_past_season', 'circuit_training_past_month', 'circuit_training_past_season', 'barefoot_past_month', 
        'barefoot_past_season', 'stretching_past_month', 'stretching_past_season', 'SC_past_month', 'non_running_past_month'
    ]
}

# Define predictors and outcome
X = data.drop(columns=['RRI'])  # Predictors
y = data['RRI']  # Outcome

# Ensure that the feature groups exist in the dataset
for group in feature_groups:
    feature_groups[group] = [feature for feature in feature_groups[group] if feature in X.columns]

num_features = len(X)
feature_to_idx = {feature: idx for idx, feature in enumerate(X.columns)}  # Corrected to X.columns

# Initialize adjacency matrix with zeros
adjacency = np.zeros((num_features, num_features), dtype=np.float32)

# Define groups
genotype_features = feature_groups['Genotype']
history_features = feature_groups['History']
phenotype_features = feature_groups['Phenotype']
behaviour_features = feature_groups['Behaviour']

# Get indices for each group
genotype_indices = [feature_to_idx[feat] for feat in genotype_features]
history_indices = [feature_to_idx[feat] for feat in history_features]
phenotype_indices = [feature_to_idx[feat] for feat in phenotype_features]
behaviour_indices = [feature_to_idx[feat] for feat in behaviour_features]

# Genotype nodes: connect to every node except themselves
for i in genotype_indices:
    for j in range(num_features):
        if j != i:
            adjacency[i, j] = 1.0

# History nodes: connect to every node except genotype nodes and themselves
for i in history_indices:
    for j in range(num_features):
        if j not in genotype_indices and j != i:
            adjacency[i, j] = 1.0

# Phenotype nodes: connect to every node except genotype, history nodes, and themselves
for i in phenotype_indices:
    for j in range(num_features):
        if j not in genotype_indices and j not in history_indices and j != i:
            adjacency[i, j] = 1.0

# Behaviour nodes: connect to all other behaviour nodes except themselves
for i in behaviour_indices:
    for j in behaviour_indices:
        if j != i:
            adjacency[i, j] = 1.0

# Convert adjacency to tensor
adjacency_mask = torch.tensor(adjacency, dtype=torch.float32)

class CustomDataset(Dataset):
    def __init__(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.X = X.values.astype(np.float32)
        elif isinstance(X, np.ndarray):
            self.X = X.astype(np.float32)
        else:
            raise TypeError("X should be a pandas DataFrame or a NumPy array.")
        self.y = y.astype(np.float32)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class CustomNetwork(nn.Module):
    def __init__(self, num_features, adjacency_mask):
        super(CustomNetwork, self).__init__()
        self.num_features = num_features

        # Register the fixed adjacency mask as a buffer (non-trainable)
        self.register_buffer('mask', adjacency_mask.float())  # Shape: (num_features, num_features)

        # Initialize connection weights (W_ij: from node j to node i)
        self.W = nn.Parameter(torch.randn(num_features, num_features) * 0.01)

        # Initialize bias for each node
        self.bias = nn.Parameter(torch.zeros(num_features))

        # Initialize output weights (from each node to the final output)
        self.output_weights = nn.Parameter(torch.randn(num_features) * 0.01)

        # Add Batch Normalization
        self.batch_norm = nn.BatchNorm1d(num_features)

    def forward(self, x, threshold=None):
        """
        x: input tensor of shape (batch_size, num_features)
        threshold: if provided, apply binary thresholding to gating
        """
        # Use the fixed mask directly
        learnable_mask = self.mask

        # Apply the learnable mask to connection weights
        W_masked = self.W * learnable_mask  # Shape: (num_features, num_features)

        # Compute incoming messages: (batch_size, num_features) @ (num_features, num_features) = (batch_size, num_features)
        incoming = torch.matmul(x, W_masked)

        # Element-wise multiplication with own feature value
        node_input = incoming * x  # Shape: (batch_size, num_features)

        node_input = self.batch_norm(node_input)

        # Add bias
        node_input += self.bias  # Broadcasting over batch

        # Apply Leaky ReLU activation
        node_output = F.leaky_relu(node_input)  # Shape: (batch_size, num_features)

        # Aggregate node outputs to final output
        output = torch.matmul(node_output, self.output_weights)  # Shape: (batch_size,)

        return output  # Raw scores (logits)

# Removed ReliefF import since it's no longer used
# from skrebate import ReliefF

def train_evaluate_fold(fold, train_index, val_index, X, y, selected_features, adjacency, feature_to_idx, 
                       n_epochs, lr, weight_decay, batch_size, threshold):
    """
    Function to train and evaluate the model on a single fold.
    This function is intended to be run in parallel.
    """
    # Split data
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # Encode labels for binary classification
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_val_enc = le.transform(y_val)
    y_train_final = y_train_enc
    y_val_final = y_val_enc

    # Create datasets
    train_dataset = CustomDataset(X_train[selected_features], y_train_final)
    val_dataset = CustomDataset(X_val[selected_features], y_val_final)

    # Create dataloaders with drop_last=True for training and drop_last=False for validation
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

    # Map selected_features to their indices
    selected_feature_indices = [feature_to_idx[feat] for feat in selected_features]

    # Select the corresponding adjacency submatrix using integer indices
    selected_adjacency = adjacency[np.ix_(selected_feature_indices, selected_feature_indices)]
    selected_adjacency_mask = torch.tensor(selected_adjacency, dtype=torch.float32)

    device = torch.device('cpu')

    # Initialize the model with the number of selected features and the selected adjacency mask
    model = CustomNetwork(num_features=len(selected_features), 
                          adjacency_mask=selected_adjacency_mask).to(device)

    # Define loss and optimizer
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Training Loop
    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device).float()  # Ensure float type
            batch_y = batch_y.to(device).float()  # BCEWithLogitsLoss expects float targets

            optimizer.zero_grad()
            outputs = model(batch_X)  # Forward pass
            loss = criterion(outputs, batch_y)

            # Total loss
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * batch_X.size(0)

        epoch_loss /= len(train_loader.dataset)

    # Validation
    model.eval()
    all_outputs = []
    all_labels = []
    all_preds = []  # Initialize a list to accumulate predictions

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(device).float()

            outputs = model(batch_X)  # Forward pass
            probs = torch.sigmoid(outputs).cpu().numpy()
            preds = (probs >= threshold).astype(int).flatten()

            all_outputs.extend(probs)
            all_labels.extend(batch_y.numpy())
            all_preds.extend(preds)  # Accumulate predictions

    # Compute metrics
    roc_auc = roc_auc_score(all_labels, all_outputs)
    f1 = f1_score(all_labels, all_preds, average='binary')
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='binary')

    return {
        'roc_auc': roc_auc,
        'f1': f1,
        'accuracy': accuracy,
        'precision': precision
    }

def global_feature_ranking(X, y):
    """
    Perform global feature ranking using Logistic Regression with L1 regularization (Lasso).
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import LabelEncoder

    # Encode labels
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    # Initialize Logistic Regression with L1 penalty and C=25
    clf = LogisticRegression(penalty='l1', C=25, solver='saga', max_iter=10000, random_state=42)
    clf.fit(X, y_encoded)

    # Get absolute coefficients as feature scores
    feature_scores = np.abs(clf.coef_[0])

    # Create a DataFrame
    feature_score_df = pd.DataFrame({
        'feature': X.columns,
        'score': feature_scores
    })

    # Sort features by score in descending order
    feature_score_df = feature_score_df.sort_values(by='score', ascending=False)

    return feature_score_df

def objective(trial, X, y, feature_groups, feature_to_idx, global_feature_score_df, adjacency, threshold=0.5):
    # ====================
    # Hyperparameters to Tune
    # ====================

    # 1. Number of Epochs
    n_epochs = trial.suggest_int('n_epochs', 500, 3000)

    # 2. Learning Rate
    lr = trial.suggest_loguniform('lr', 1e-6, 1e-2)

    # 3. Weight Decay
    weight_decay = trial.suggest_loguniform('weight_decay', 1e-8, 1e-3)

    # 4. Batch Size
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256, 512])

    # 5. Number of Features per Group
    n_genotype = trial.suggest_int('n_genotype', 1, len(feature_groups['Genotype']))
    n_history = trial.suggest_int('n_history', 1, len(feature_groups['History']))
    n_phenotype = trial.suggest_int('n_phenotype', 1, len(feature_groups['Phenotype']))
    n_behaviour = trial.suggest_int('n_behaviour', 1, len(feature_groups['Behaviour']))

    # ====================
    # Feature Selection Using Global Rankings
    # ====================

    # Initialize selected features list
    selected_features = []

    # For each group, select the top N features based on global Logistic Lasso scores
    for group, n_select in zip(['Genotype', 'History', 'Phenotype', 'Behaviour'],
                               [n_genotype, n_history, n_phenotype, n_behaviour]):
        group_features = feature_groups[group]
        # Filter the features present in the group
        group_feature_scores = global_feature_score_df[global_feature_score_df['feature'].isin(group_features)]
        # Select top N features
        top_features = group_feature_scores.head(n_select)['feature'].tolist()
        selected_features.extend(top_features)

    # Ensure that selected_features are unique
    selected_features = list(dict.fromkeys(selected_features))

    # ====================
    # Data Preparation
    # ====================

    X_selected = X[selected_features].copy()
    y_selected = y.copy()

    # Initialize Stratified K-Fold
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    # Get all fold indices
    fold_indices = list(skf.split(X_selected, y_selected))

    # ====================
    # Parallel Training and Evaluation
    # ====================

    # Define a wrapper for joblib's Parallel
    results = Parallel(n_jobs=10)(
        delayed(train_evaluate_fold)(
            fold=fold,
            train_index=train_idx,
            val_index=val_idx,
            X=X_selected,
            y=y_selected,
            selected_features=selected_features,
            adjacency=adjacency,  # Pass the numpy adjacency matrix
            feature_to_idx=feature_to_idx,  # Pass the feature_to_idx mapping
            n_epochs=n_epochs,
            lr=lr,
            weight_decay=weight_decay,
            batch_size=batch_size,
            threshold=threshold
        )
        for fold, (train_idx, val_idx) in enumerate(fold_indices)
    )

    # ====================
    # Collect Metrics
    # ====================

    aucs = []
    f1s = []
    accuracies = []
    precisions = []

    for res in results:
        aucs.append(res['roc_auc'])
        f1s.append(res['f1'])
        accuracies.append(res['accuracy'])
        precisions.append(res['precision'])

    # ====================
    # Final Metrics Calculation
    # ====================

    mean_auc = np.mean(aucs)
    std_auc = np.std(aucs)
    mean_f1 = np.mean(f1s)
    std_f1 = np.std(f1s)
    mean_accuracy = np.mean(accuracies)
    std_accuracy = np.std(accuracies)
    mean_precision = np.mean(precisions)
    std_precision = np.std(precisions)

    # ====================
    # Objective Metric
    # ====================

    # Since it's binary classification, we aim to maximize the mean AUC
    trial.set_user_attr("mean_auc", mean_auc)
    trial.set_user_attr("std_auc", std_auc)
    trial.set_user_attr("mean_f1", mean_f1)
    trial.set_user_attr("std_f1", std_f1)
    trial.set_user_attr("mean_accuracy", mean_accuracy)
    trial.set_user_attr("std_accuracy", std_accuracy)
    trial.set_user_attr("mean_precision", mean_precision)
    trial.set_user_attr("std_precision", std_precision)
    trial.set_user_attr("selected_features", selected_features)

    return mean_auc

def run_hyperparameter_tuning(X, y, feature_groups, feature_to_idx, adjacency, threshold=0.5):
    # Generate the global feature ranking before the study starts using Logistic Lasso
    global_feature_score_df = global_feature_ranking(X, y)

    def objective_wrapper(trial):
        return objective(trial, X, y, feature_groups, feature_to_idx, global_feature_score_df, adjacency, threshold)

    study = optuna.create_study(
        direction='maximize',
        study_name='CustomNetwork_Hyperparameter_Tuning',
        sampler=optuna.samplers.TPESampler(seed=42)
    )

    # Integrate a progress bar for the Optuna study
    with tqdm(total=500, desc="Optuna Trials") as pbar:
        def callback(study, trial):
            pbar.update(1)

        # Set n_jobs=1 to ensure serial execution of hyperparameter trials
        study.optimize(objective_wrapper, n_trials=500, timeout=None, callbacks=[callback], n_jobs=1)

    print("Number of finished trials: ", len(study.trials))
    print("Best trial:")
    trial = study.best_trial

    print("  Value (Mean AUC): ", trial.value)
    print("  Params: ")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")

    # Extract additional metrics from the best trial
    best_trial_metrics = trial.user_attrs
    mean_auc = best_trial_metrics.get("mean_auc", None)
    std_auc = best_trial_metrics.get("std_auc", None)
    mean_f1 = best_trial_metrics.get("mean_f1", None)
    std_f1 = best_trial_metrics.get("std_f1", None)
    mean_accuracy = best_trial_metrics.get("mean_accuracy", None)
    std_accuracy = best_trial_metrics.get("std_accuracy", None)
    mean_precision = best_trial_metrics.get("mean_precision", None)
    std_precision = best_trial_metrics.get("std_precision", None)
    selected_features = best_trial_metrics.get("selected_features", None)

    print(f"\nBest Trial Metrics:")
    print(f"Selected Features: {selected_features}")
    print(f"Average AUC: {mean_auc:.4f} ± {std_auc:.4f}")
    print(f"Average F1 Score: {mean_f1:.4f} ± {std_f1:.4f}")
    print(f"Average Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")
    print(f"Average Precision: {mean_precision:.4f} ± {std_precision:.4f}")

    return study

if __name__ == "__main__":
    # Run hyperparameter tuning
    study = run_hyperparameter_tuning(X, y, feature_groups, feature_to_idx, adjacency)

[I 2024-12-17 00:10:02,994] A new study created in memory with name: CustomNetwork_Hyperparameter_Tuning
Optuna Trials:  19%|█▉        | 97/500 [8:33:00<48:27:31, 432.88s/it]/home/h_wu4/ml_env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/h_wu4/ml_env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/h_wu4/ml_env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control th

Number of finished trials:  500
Best trial:
  Value (Mean AUC):  0.7503470913074771
  Params: 
    n_epochs: 1142
    lr: 3.5784877937744905e-05
    weight_decay: 8.830400761450904e-08
    batch_size: 128
    n_genotype: 54
    n_history: 6
    n_phenotype: 12
    n_behaviour: 7

Best Trial Metrics:
Selected Features: ['rs7035322', 'rs145648292', 'rs1330363', 'rs144414988', 'rs9340799', 'rs1676303', 'rs2277268', 'rs1011814', 'rs4903399', 'rs1590', 'rs4986938', 'rs4919510', 'rs149047058', 'rs2289360', 'rs1800469', 'rs1249269', 'rs2281518', 'rs2586488', 'rs4454832', 'rs42522', 'rs3216902', 'rs2858056', 'rs591058', 'rs4362400', 'rs10132091', 'rs1937810', 'rs2306033', 'rs1643821', 'rs10484958', 'rs1045485', 'rs42517', 'rs4701616', 'rs2525504', 'rs761804508', 'rs1134170', 'rs11225395', 'rs10759753', 'sex', 'rs1800629', 'rs1144393', 'rs3045', 'rs17583842', 'rs1800972', 'rs4328262', 'rs2234693', 'rs3751143', 'rs1800470', 'rs4988321', 'rs2228570', 'rs78391032', 'rs12656106', 'rs60713544', 'rs1